In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from torch.nn.functional import cosine_similarity
from sklearn.decomposition import PCA

/Users/jadfaddil/Desktop/School Work/Pole projet/new_pole_projet_s7/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
FIG_DIR = "figures"
INPUT_FILE = "text.txt"

os.makedirs(FIG_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# 3. Load model & tokenizer
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    output_hidden_states=True,
    output_attentions=True,
    device_map="auto"
)

model.eval()


`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['output_attentions', 'output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Some parameters are on the meta device because they were offloaded to the disk.


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

In [6]:
attn_outputs = {} 

def make_attn_hook(layer_id):
    def hook(module, inp, out):
        # out can be a tensor or a tuple
        if isinstance(out, tuple):
            attn_outputs[layer_id] = out[0].detach().cpu()
        else:
            attn_outputs[layer_id] = out.detach().cpu()
    return hook

for layer_id, layer in enumerate(model.model.layers):
    layer.self_attn.register_forward_hook(make_attn_hook(layer_id))

In [7]:
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    text = f.read().strip()

max_len = 256
inputs = tokenizer(text, return_tensors="pt",truncation=True,max_length=max_len).to(device)

In [8]:
with torch.no_grad():
    outputs = model(**inputs)

In [9]:
hidden_states = [h.squeeze(0).cpu() for h in outputs.hidden_states]
attentions = outputs.attentions
L = len(hidden_states)
seq_len = hidden_states[0].shape[0]

In [10]:
var = {}
for i in range(16):
    var[i] = np.cov(attn_outputs[i].squeeze(0).numpy(), rowvar=False, bias=False)

In [11]:
eigvals = {}
eigvecs = {}
for i in range(16):
    eigvals[i], eigvecs[i] = np.linalg.eigh(var[i])
    idx = np.argsort(eigvals[i])[::-1]
    eigvals[i] = eigvals[i][idx]
    eigvecs[i]= eigvecs[i][:, idx]

In [12]:
EIG_DIR = os.path.join(FIG_DIR, "eigvals")

os.makedirs(EIG_DIR, exist_ok=True)

for layer_id, eig in eigvals.items():
    plt.figure(figsize=(6,4))
    
    eig_to_plot = eig[:seq_len-1]
    
    plt.plot(eig_to_plot, marker='o', linestyle='-', markersize=3)
    plt.yscale('log')  # log scale for better visualization
    plt.xlabel("Eigenvalue index")
    plt.ylabel("Eigenvalue (log scale)")
    plt.title(f"Layer {layer_id} Eigenvalues (top 256)")
    plt.grid(True, which="both", ls="--", alpha=0.5)

    fname = os.path.join(EIG_DIR, f"layer_{layer_id}_eigvals.png")
    plt.savefig(fname, bbox_inches='tight')
    plt.close()


In [16]:
COS_DIR = os.path.join(FIG_DIR, "cosine")
os.makedirs(COS_DIR, exist_ok=True)

for layer_id, X in attn_outputs.items():

    X = X.squeeze(0).numpy()
    X = X - X.mean(axis=0, keepdims=True)
    X_norm = X / np.linalg.norm(X, axis=1, keepdims=True)

    cos_sim = X_norm @ X_norm.T
    
    plt.figure(figsize=(6,5))
    plt.imshow(cos_sim, cmap="viridis", vmin=-1, vmax=1)
    plt.colorbar(label="Cosine similarity")
    plt.title(f"Layer {layer_id} – Cosine Similarity (Attention Output)")
    plt.xlabel("Token index")
    plt.ylabel("Token index")
    plt.tight_layout()
    
    fname = os.path.join(COS_DIR, f"layer_{layer_id}_cosine.png")
    plt.savefig(fname, bbox_inches="tight")
    plt.close()

In [18]:
def select_pca_dim(eigvals_layer, p=0.9):
    total_var = np.sum(eigvals_layer)
    cum_var = np.cumsum(eigvals_layer)
    k_layer = np.searchsorted(cum_var / total_var, p) + 1
    return k_layer

k = {}
for layer_id, vals in eigvals.items():
    k[layer_id] = select_pca_dim(vals, p=0.9)

In [19]:
W = {}
for i in range(16):
    X = attn_outputs[layer_id].squeeze(0).numpy()
    mu = X.mean(axis=0, keepdims=True)
    X_centered = X - mu
    V_k = eigvecs[i][:, :seq_len-1]
    W[i] = X_centered @ V_k

In [20]:
COS_PROJ_DIR_SEQ_LEN = os.path.join(FIG_DIR, "cosine_proj_seq_len")
os.makedirs(COS_PROJ_DIR_SEQ_LEN, exist_ok=True)

for layer_id, W_layer in W.items():

    W_norm = W_layer / np.linalg.norm(W_layer, axis=1, keepdims=True)
    cos_sim = W_norm @ W_norm.T
    
    plt.figure(figsize=(6,5))
    plt.imshow(cos_sim, cmap="viridis", vmin=-1, vmax=1)
    plt.colorbar(label="Cosine similarity")
    plt.title(f"Layer {layer_id} – Cosine Similarity (Projected for seq_len)")
    plt.xlabel("Token index")
    plt.ylabel("Token index")
    plt.tight_layout()

    fname = os.path.join(COS_PROJ_DIR_SEQ_LEN, f"layer_{layer_id}_cosine_proj_seq_len.png")
    plt.savefig(fname, bbox_inches="tight")
    plt.close()

In [21]:
Z = {}
for i in range(16):
    X = attn_outputs[layer_id].squeeze(0).numpy()
    mu = X.mean(axis=0, keepdims=True)
    X_centered = X - mu
    V_k = eigvecs[i][:, :k[i]]
    Z[i] = X_centered @ V_k

In [22]:
COS_PROJ_DIR = os.path.join(FIG_DIR, "cosine_proj")
os.makedirs(COS_PROJ_DIR, exist_ok=True)

for layer_id, Z_layer in Z.items():

    Z_norm = Z_layer / np.linalg.norm(Z_layer, axis=1, keepdims=True)
    cos_sim = Z_norm @ Z_norm.T
    
    plt.figure(figsize=(6,5))
    plt.imshow(cos_sim, cmap="viridis", vmin=-1, vmax=1)
    plt.colorbar(label="Cosine similarity")
    plt.title(f"Layer {layer_id} – Cosine Similarity (Projected)")
    plt.xlabel("Token index")
    plt.ylabel("Token index")
    plt.tight_layout()

    fname = os.path.join(COS_PROJ_DIR, f"layer_{layer_id}_cosine_proj.png")
    plt.savefig(fname, bbox_inches="tight")
    plt.close()